In [ ]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold


# ---------- Scaffold ----------
def bemis_murcko_scaffold(smiles: str) -> str | None:
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    if scaf is None:
        return None
    return Chem.MolToSmiles(scaf, isomericSmiles=False)


# ---------- Labeling ----------
def label_from_pchembl(p: float, pos_thr=6.0, neg_thr=5.0) -> int | None:
    """
    Returns:
      1 for positive (active)
      0 for negative (inactive)
      None for ambiguous (drop)
    """
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return None
    if p >= pos_thr:
        return 1
    if p <= neg_thr:
        return 0
    return None


# ---------- Scaffold split ----------
def scaffold_split(df: pd.DataFrame, smiles_col="smiles", test_size=0.2, seed=42):
    df = df.copy()
    df["scaffold"] = df[smiles_col].apply(bemis_murcko_scaffold)
    df = df.dropna(subset=["scaffold"]).reset_index(drop=True)

    scaffold_to_indices = df.groupby("scaffold").indices
    scaffolds = list(scaffold_to_indices.keys())

    rng = np.random.default_rng(seed)
    rng.shuffle(scaffolds)

    n_total = len(df)
    n_test_target = int(round(test_size * n_total))

    test_scaffolds = []
    test_count = 0

    for scaf in scaffolds:
        idxs = scaffold_to_indices[scaf]
        if test_count + len(idxs) <= n_test_target:
            test_scaffolds.append(scaf)
            test_count += len(idxs)

    is_test = df["scaffold"].isin(test_scaffolds)

    train_df = df.loc[~is_test].drop(columns=["scaffold"]).reset_index(drop=True)
    test_df  = df.loc[ is_test].drop(columns=["scaffold"]).reset_index(drop=True)
    return train_df, test_df


def scaffold_overlap(train_df, test_df, smiles_col="smiles") -> int:
    tr = set(train_df[smiles_col].apply(bemis_murcko_scaffold).dropna())
    te = set(test_df[smiles_col].apply(bemis_murcko_scaffold).dropna())
    return len(tr.intersection(te))


# ---------- Example end-to-end ----------
def make_controls_and_split(df_long: pd.DataFrame,
                            smiles_col="smiles",
                            target_col="target",          # e.g. "CHRM3"
                            pchembl_col="pchembl_value",
                            pos_thr=6.0,
                            neg_thr=5.0,
                            test_size=0.2,
                            seed=42):
    """
    df_long expected columns: smiles, target, pchembl_value
    returns: df_train, df_test with a 'y' label column
    """
    df = df_long.copy()

    # Keep only rows with usable SMILES/pChEMBL
    df = df.dropna(subset=[smiles_col, pchembl_col]).reset_index(drop=True)

    # Assign y labels with drop-zone
    df["y"] = df[pchembl_col].apply(lambda p: label_from_pchembl(p, pos_thr=pos_thr, neg_thr=neg_thr))
    df = df.dropna(subset=["y"]).reset_index(drop=True)
    df["y"] = df["y"].astype(int)

    # Optional: keep only one measurement per compound per target
    # (recommended: take max pChEMBL for "is active?" tasks; for inactivity you might take min;
    # simplest robust rule for binary activity is max)
    df = (df.sort_values(pchembl_col, ascending=False)
            .groupby([smiles_col, target_col], as_index=False)
            .head(1)
            .reset_index(drop=True))

    train_df, test_df = scaffold_split(df, smiles_col=smiles_col, test_size=test_size, seed=seed)

    ov = scaffold_overlap(train_df, test_df, smiles_col=smiles_col)
    print(f"[CHECK] Scaffold overlap = {ov} (must be 0)")

    # Basic stats
    def stats(x):
        return x["y"].value_counts().to_dict()

    print("[STATS] train y:", stats(train_df))
    print("[STATS] test  y:", stats(test_df))

    return train_df, test_df
